<a href="https://colab.research.google.com/github/SuyaNangabo/plp-da-a3-duka-sales-and-agent-cleanup/blob/main/duka_sales_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Duka Electronics Half-Year Sales Review

Exploring Duka Electronics sales for January to June 2026 to answer the Head of Sales' questions ahead of the half-year review.

## Executive Summary

- Total revenue for January to June 2026 was KES 71.8M from 1,000 orders, and Nairobi contributed KES 26.1M, which is 36.4% of the total and almost double Mombasa in second place (KES 14.0M).
- Laptops generated KES 41.5M, more than double the next product (Tablets at KES 19.3M), even though Tablets sold more units (691 against 639).
- Weekend orders are worth the same as weekday orders on average (KES 71,972 against KES 71,724), so there is no case for weekend-only promotions based on basket size.
- April was the strongest month at KES 13.5M, 24.0% above June, the weakest month at KES 10.9M.
- Premium online orders from Nairobi and Mombasa came to 83 orders worth KES 10.2M, which makes them a clear target group for online marketing.

## Setup

In [14]:
import os
import numpy as np
import pandas as pd

os.makedirs("outputs", exist_ok=True)

## Create the Data

This cell saves `generate_data.py`, which contains both dataset generators from this week unchanged, and the next cell runs it to create the raw CSVs.

In [15]:
%%writefile generate_data.py

import numpy as np
import pandas as pd

# Engagement 1: Duka Electronics (seed 2026)
rng = np.random.default_rng(2026)
n = 1000
products = {"Phone": 15000, "Laptop": 65000, "Headphones": 2500,
            "Tablet": 28000, "Charger": 1200}

sales = pd.DataFrame({
    "order_id": range(1001, 1001 + n),
    "order_date": pd.to_datetime("2026-01-01")
                  + pd.to_timedelta(rng.integers(0, 180, n), unit="D"),
    "branch": rng.choice(["Nairobi", "Mombasa", "Kisumu", "Nakuru", "Eldoret"], n,
                         p=[0.4, 0.2, 0.15, 0.15, 0.1]),
    "product": rng.choice(list(products), n),
    "quantity": rng.integers(1, 6, n),
    "channel": rng.choice(["In-store", "Online", "Phone order"], n,
                          p=[0.55, 0.35, 0.10]),
})
sales["unit_price"] = sales["product"].map(products)
sales.to_csv("duka_sales.csv", index=False)

# Engagement 2: Mobile money agent network (seed 44)
rng = np.random.default_rng(44)
n = 600

agents = pd.DataFrame({
    "agent_id": [f"AG{i:03d}" for i in range(1, 21)],
    "county": rng.choice(["Nairobi", "Kiambu", "Nakuru", "Mombasa", "Kisumu"], 20),
    "onboarded": pd.date_range("2024-01-01", periods=20, freq="MS")
                   .strftime("%Y-%m-%d"),
})

txns = pd.DataFrame({
    "txn_id": [f"TX{i:05d}" for i in range(1, n + 1)],
    "agent_id": rng.choice(agents["agent_id"], n),
    "txn_type": rng.choice(["Deposit", "withdrawal", "DEPOSIT ",
                            "Withdrawal", "deposit"], n),
    "amount": rng.lognormal(7.5, 0.8, n).round(0).astype(str),
    "txn_date": (pd.to_datetime("2026-03-01")
                 + pd.to_timedelta(rng.integers(0, 60, n), unit="D"))
                .strftime("%Y-%m-%d"),
})

idx = rng.choice(n, 30, replace=False)
txns.loc[idx[:15], "amount"] = np.nan
txns.loc[idx[15:20], "amount"] = "N/A"
txns.loc[idx[20:25], "amount"] = "250000"
txns.loc[idx[25:], "txn_date"] = txns.loc[idx[25:], "txn_date"].str.replace("-", "/")
txns = pd.concat([txns, txns.sample(12, random_state=1)], ignore_index=True)
txns.loc[rng.choice(len(txns), 8, replace=False), "agent_id"] = "AG099"

agents.to_csv("agents.csv", index=False)
txns.to_csv("agent_transactions.csv", index=False)

print("Created duka_sales.csv, agents.csv and agent_transactions.csv")

Overwriting generate_data.py


In [16]:
%run generate_data.py


Created duka_sales.csv, agents.csv and agent_transactions.csv


## Task 1: First-Five-Minutes Inspection

In [17]:
sales = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])

print("Shape:", sales.shape)
print()
sales.info()

Shape: (1000, 7)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   order_id    1000 non-null   int64         
 1   order_date  1000 non-null   datetime64[ns]
 2   branch      1000 non-null   object        
 3   product     1000 non-null   object        
 4   quantity    1000 non-null   int64         
 5   channel     1000 non-null   object        
 6   unit_price  1000 non-null   int64         
dtypes: datetime64[ns](1), int64(3), object(3)
memory usage: 54.8+ KB


In [18]:
sales.head()

,order_id,order_date,branch,product,quantity,channel,unit_price
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500


In [19]:
print("Missing values per column")
print(sales.isna().sum())
print()
print("First order:", sales["order_date"].min().date())
print("Last order: ", sales["order_date"].max().date())
print()
print(sales["branch"].value_counts())
print()
print(sales["product"].value_counts())
print()
print(sales["channel"].value_counts(normalize=True).round(3))

Missing values per column
order_id      0
order_date    0
branch        0
product       0
quantity      0
channel       0
unit_price    0
dtype: int64

First order: 2026-01-01
Last order:  2026-06-29

branch
Nairobi    376
Mombasa    186
Nakuru     162
Kisumu     159
Eldoret    117
Name: count, dtype: int64

product
Laptop        220
Tablet        215
Phone         194
Headphones    187
Charger       184
Name: count, dtype: int64

channel
In-store       0.579
Online         0.320
Phone order    0.101
Name: proportion, dtype: float64


### Inspection Notes

- **Shape:** 1,000 orders and 7 columns.
- **Types:** order_date loaded as a real date, order_id, quantity and unit_price are whole numbers, and branch, product and channel are text.
- **Missing values:** none in any column.
- **Date range:** 1 January 2026 to 29 June 2026, so June is one day short of a full month.
- **Branches:** Nairobi has the most orders (376), followed by Mombasa (186), Nakuru (162), Kisumu (159) and Eldoret (117).
- **Products:** order counts are spread evenly, from Charger (184) to Laptop (220).
- **Channels:** 57.9% of orders are in-store, 32.0% online and 10.1% by phone.

## Task 2: Derived Columns

In [20]:
sales["revenue"] = sales["quantity"] * sales["unit_price"]
sales["month"] = sales["order_date"].dt.month_name()
sales["weekday"] = sales["order_date"].dt.day_name()
sales["is_weekend"] = sales["order_date"].dt.dayofweek >= 5
sales["price_band"] = np.where(sales["unit_price"] >= 28000, "Premium", "Value")

sales.head()

,order_id,order_date,branch,product,quantity,channel,unit_price,revenue,month,weekday,is_weekend,price_band
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000,45000,June,Wednesday,False,Value
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000,260000,February,Monday,False,Premium
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000,140000,January,Monday,False,Premium
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500,12500,April,Sunday,True,Value
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500,7500,March,Saturday,True,Value


## Task 3: Branch Rankings by Revenue and Orders

In [21]:
branches = sales.groupby("branch").agg(
    total_revenue=("revenue", "sum"),
    orders=("order_id", "count"),
    avg_order=("revenue", "mean"),
)
branches["revenue_rank"] = branches["total_revenue"].rank(ascending=False).astype(int)
branches["orders_rank"] = branches["orders"].rank(ascending=False).astype(int)
branches["avg_order"] = branches["avg_order"].round(0)

branches.sort_values("revenue_rank")

,total_revenue,orders,avg_order,revenue_rank,orders_rank
branch,,,,,
Nairobi,26106100,376,69431.0,1,1
Mombasa,14003700,186,75289.0,2,2
Nakuru,11970400,162,73891.0,3,3
Kisumu,10868100,159,68353.0,4,4
Eldoret,8844200,117,75591.0,5,5


**Are the rankings the same?** Yes. Nairobi, Mombasa, Nakuru, Kisumu and Eldoret are in the same order for both revenue and number of orders. The average order value is similar in every branch, between KES 68,353 and KES 75,591, so the branch with more orders also earns more. Eldoret has the highest average order (KES 75,591) but it has far fewer orders than the others, so it still comes last on revenue.

## Task 4: Top Products by Revenue and Units Sold

In [22]:
products_summary = sales.groupby("product").agg(
    revenue=("revenue", "sum"),
    units=("quantity", "sum"),
)

print("Top 3 by revenue")
print(products_summary["revenue"].nlargest(3))
print()
print("Top 3 by units sold")
print(products_summary["units"].nlargest(3))

Top 3 by revenue
product
Laptop    41535000
Tablet    19348000
Phone      8880000
Name: revenue, dtype: int64

Top 3 by units sold
product
Tablet    691
Laptop    639
Phone     592
Name: units, dtype: int64


**Why the lists differ:** Laptops earn the most revenue (KES 41.5M), but Tablets sell the most units (691 against 639 for Laptops). A laptop costs KES 65,000 and a tablet costs KES 28,000, so each laptop brings in more than twice as much money. That is why Laptops lead on revenue even though fewer units were sold. Phone is third on both lists.

## Task 5: Weekend vs Weekday Orders

In [23]:
weekend = sales.groupby("is_weekend").agg(
    orders=("order_id", "count"),
    total_revenue=("revenue", "sum"),
    avg_order=("revenue", "mean"),
).round(0)
weekend.index = weekend.index.map({False: "Weekday", True: "Weekend"})
weekend

,orders,total_revenue,avg_order
is_weekend,,,
Weekday,723,51856200,71724.0
Weekend,277,19936300,71972.0


**Weekend vs weekday:** weekdays have 723 orders and KES 51.9M revenue, and weekends have 277 orders and KES 19.9M. A weekend has 2 days out of 7, so it should get about 28.6% of orders, and it gets 27.7%. The average order is almost the same, KES 71,724 on weekdays and KES 71,972 at weekends, so customers do not spend more at weekends.

## Task 6: Best and Worst Month

In [24]:
monthly = sales.groupby(sales["order_date"].dt.to_period("M"))["revenue"].sum()
monthly.index = monthly.index.strftime("%B")
print(monthly)
print()

best_month = monthly.idxmax()
worst_month = monthly.idxmin()
pct_diff = (monthly.max() - monthly.min()) / monthly.min() * 100

print(f"Best month:  {best_month} (KES {monthly.max():,})")
print(f"Worst month: {worst_month} (KES {monthly.min():,})")
print(f"The best month is {pct_diff:.1f}% higher than the worst month")

order_date
January     11786600
February    11772000
March       12396800
April       13518300
May         11415400
June        10903400
Name: revenue, dtype: int64

Best month:  April (KES 13,518,300)
Worst month: June (KES 10,903,400)
The best month is 24.0% higher than the worst month


**Best and worst month:** April was the best month at KES 13,518,300 and June was the worst at KES 10,903,400. April revenue is 24.0% higher than June. Part of the gap is because the data stops on 29 June, so June has one day less than April.

## Task 7: Premium Online Orders from Nairobi and Mombasa

In [25]:
premium_online = sales[
    (sales["price_band"] == "Premium")
    & (sales["channel"] == "Online")
    & (sales["branch"].isin(["Nairobi", "Mombasa"]))
]

print("Number of orders:", len(premium_online))
print(f"Total revenue: KES {premium_online['revenue'].sum():,}")

premium_online.to_csv("outputs/premium_online_coast_capital.csv", index=False)
print("Saved outputs/premium_online_coast_capital.csv")

Number of orders: 83
Total revenue: KES 10,218,000
Saved outputs/premium_online_coast_capital.csv


**Answer:** there are 83 Premium online orders from Nairobi and Mombasa, worth KES 10,218,000 in total. They are saved in `outputs/premium_online_coast_capital.csv` without the index.

## Task 8: Spreadsheet Check

The branch ranking was rebuilt in Google Sheets with SUMIFS, a region column was added with XLOOKUP and a PivotTable of revenue by branch and channel was built. The file is saved as `excel/duka_branch_summary.xlsx`. The cell below reads the SUMIFS totals from the Summary sheet and compares them with pandas.

In [26]:
excel_path = "excel/duka_branch_summary.xlsx"

if os.path.exists(excel_path):
    excel_summary = pd.read_excel(excel_path, sheet_name="Summary")
    excel_summary = excel_summary.set_index("branch")["revenue"]

    check = pd.DataFrame({
        "pandas": sales.groupby("branch")["revenue"].sum(),
        "excel": excel_summary,
    })
    check["match"] = check["pandas"] == check["excel"]
    print(check)
    print()
    print("All totals match:", check["match"].all())
else:
    print("Upload excel/duka_branch_summary.xlsx and run this cell again")

           pandas     excel  match
branch                            
Eldoret   8844200   8844200   True
Kisumu   10868100  10868100   True
Mombasa  14003700  14003700   True
Nairobi  26106100  26106100   True
Nakuru   11970400  11970400   True

All totals match: True
